In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/muhammmadsarmad/eye-disease-mendeley/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation/Eyelid/266.jpeg
/kaggle/input/datasets/muhammmadsarmad/eye-disease-mendeley/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation/Eyelid/228.jpeg
/kaggle/input/datasets/muhammmadsarmad/eye-disease-mendeley/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation/Eyelid/761.jpeg
/kaggle/input/datasets/muhammmadsarmad/eye-disease-mendeley/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation/Eyelid/833.jpeg
/kaggle/input/datasets/muhammmadsarmad/eye-disease-mendeley/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation/Eyelid/130.jpeg


In [2]:
# Step 1: Import libraries for file handling, hashing, image processing, OCR, and splitting
import os, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError, ExifTags
from sklearn.model_selection import train_test_split

!pip install imagehash pytesseract -q
!apt-get install -y tesseract-ocr -q > /dev/null
import imagehash
import pytesseract

In [3]:
# Step 2: Auto-detect dataset root folder that directly contains the disease class subfolders
def find_data_root():
    kaggle_input = Path("/kaggle/input")
    expected_classes = {"Cataract", "Conjunctivitis", "Eyelid", "Normal", "Uveitis"}
    for p in kaggle_input.rglob("*"):
        if p.is_dir():
            subdirs = {c.name for c in p.iterdir() if c.is_dir()}
            if expected_classes.issubset(subdirs):
                print(f"Found dataset at: {p}")
                return p
    print("[WARN] Not found. Contents of /kaggle/input:")
    for p in kaggle_input.iterdir():
        print(" -", p)
    return None

DATA_ROOT = find_data_root()

Found dataset at: /kaggle/input/datasets/muhammmadsarmad/eye-disease-mendeley/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation


In [4]:
# Step 3: Set up classes, label mapping, output path, image size, dedup threshold, and watermark keywords
CLASSES = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
LABEL_MAP = {name: i for i, name in enumerate(CLASSES)}
OUTPUT_DIR = Path("/kaggle/working/preprocessed_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE = (224, 224)
PHASH_THRESHOLD = 5

# Keywords commonly found in stock-photo watermarks or web-downloaded screenshots
WATERMARK_KEYWORDS = [
    "shutterstock", "istock", "getty", "gettyimages", "alamy", "dreamstime",
    "123rf", "depositphotos", "adobe stock", "stock photo", "watermark",
    "google", "sample", "preview", "www.", ".com"
]

print(LABEL_MAP)

{'Cataract': 0, 'Conjunctivitis': 1, 'Eyelid': 2, 'Normal': 3, 'Uveitis': 4}


In [5]:
# Step 4: Build a dataframe listing every image's path, class, and label (split will be assigned later)
records = []
for cls in CLASSES:
    d = DATA_ROOT / cls
    for fname in sorted(os.listdir(d)):
        fpath = d / fname
        if fpath.is_file():
            records.append({"path": str(fpath), "filename": fname,
                             "class_name": cls, "label": LABEL_MAP[cls]})
df = pd.DataFrame(records)
print(f"Total images: {len(df)}")
df.head()

Total images: 2298


,path,filename,class_name,label
0,/kaggle/input/datasets/muhammmadsarmad/eye-dis...,1.jpg,Cataract,0
1,/kaggle/input/datasets/muhammmadsarmad/eye-dis...,10.jpg,Cataract,0
2,/kaggle/input/datasets/muhammmadsarmad/eye-dis...,100.jpg,Cataract,0
3,/kaggle/input/datasets/muhammmadsarmad/eye-dis...,101.jpg,Cataract,0
4,/kaggle/input/datasets/muhammmadsarmad/eye-dis...,102.jpg,Cataract,0


In [6]:
# Step 5: Validate every image and drop corrupt/unreadable files before training
widths, heights, valid_flags = [], [], []
for path in df["path"]:
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            w, h = im.size
        widths.append(w); heights.append(h); valid_flags.append(True)
    except (UnidentifiedImageError, OSError):
        widths.append(None); heights.append(None); valid_flags.append(False)
df["width"] = widths
df["height"] = heights
df["valid"] = valid_flags
n_bad = (~df["valid"]).sum()
if n_bad:
    print(f"Dropping {n_bad} corrupt images")
df = df[df["valid"]].drop(columns=["valid"]).reset_index(drop=True)
print(f"Valid images: {len(df)}")

Valid images: 2298


In [7]:
# Step 6: Flag images that likely came from the web (watermark text via OCR, or screenshot signatures in EXIF)
def check_web_source(path):
    reason = None
    try:
        with Image.open(path) as im:
            # --- Check 1: OCR text scan for stock-photo / watermark keywords ---
            text = pytesseract.image_to_string(im.convert("RGB")).lower()
            for kw in WATERMARK_KEYWORDS:
                if kw in text:
                    reason = f"watermark_text:{kw}"
                    break

            # --- Check 2: EXIF 'Software' tag pointing to screenshot/browser tools ---
            if reason is None:
                exif = im.getexif()
                if exif:
                    software = str(exif.get(305, "")).lower()  # 305 = Software tag
                    if any(x in software for x in ["screenshot", "chrome", "firefox", "snip", "gimp", "photoshop"]):
                        reason = f"exif_software:{software}"
    except Exception:
        pass
    return reason

web_source_flags = []
for path in df["path"]:
    web_source_flags.append(check_web_source(path))

df["possible_web_source"] = web_source_flags
n_flagged = df["possible_web_source"].notna().sum()
print(f"Images flagged as possibly web-downloaded/watermarked: {n_flagged}")
if n_flagged:
    print(df[df["possible_web_source"].notna()][["filename", "class_name", "possible_web_source"]])

Images flagged as possibly web-downloaded/watermarked: 10
    filename class_name                                possible_web_source
43   141.jpg   Cataract        exif_software:adobe photoshop cs6 (windows)
47   145.jpg   Cataract  exif_software:adobe photoshop lightroom 5.0 (w...
48   146.jpg   Cataract        exif_software:adobe photoshop cs3 macintosh
49   147.jpg   Cataract        exif_software:adobe photoshop cs3 macintosh
67   165.jpg   Cataract          exif_software:adobe photoshop cs5 windows
82   179.jpg   Cataract  exif_software:adobe photoshop cc 2015.5 (windows)
84   180.jpg   Cataract  exif_software:adobe photoshop cc 2015.5 (windows)
92   188.jpg   Cataract        exif_software:adobe photoshop cs5.1 windows
98   193.jpg   Cataract        exif_software:adobe photoshop cs5.1 windows
122  214.jpg   Cataract         exif_software:adobe photoshop cs macintosh


In [8]:
# Step 7: Detect exact duplicate images via MD5 hash (e.g. SMOTE-copied or re-saved duplicate files)
md5_hashes = []
for path in df["path"]:
    with open(path, "rb") as f:
        md5_hashes.append(hashlib.md5(f.read()).hexdigest())
df["md5"] = md5_hashes
dup_mask = df.duplicated(subset="md5", keep="first")
print(f"Exact duplicates: {dup_mask.sum()}")
df["exact_duplicate"] = dup_mask

Exact duplicates: 1


In [9]:
# Step 8: Detect near-duplicate images via perceptual hash since MD5 misses visually similar/re-compressed images
phashes = []
for path in df["path"]:
    with Image.open(path) as im:
        phashes.append(imagehash.phash(im.convert("RGB")))
df["phash"] = phashes
near_dup_flags = [False] * len(df)
seen = []
for i, h in enumerate(df["phash"]):
    if df.loc[i, "exact_duplicate"]:
        continue
    matched = False
    for h2 in seen:
        if h - h2 <= PHASH_THRESHOLD:
            matched = True
            break
    if matched:
        near_dup_flags[i] = True
    else:
        seen.append(h)
df["near_duplicate"] = near_dup_flags
print(f"Near duplicates: {sum(near_dup_flags)}")
df = df.drop(columns=["phash"])

Near duplicates: 123


In [10]:
# Step 9: Drop any image that's an exact duplicate, near duplicate, OR flagged as a likely web/stock-photo source
df["drop_image"] = df["exact_duplicate"] | df["near_duplicate"] | df["possible_web_source"].notna()
clean = df[~df["drop_image"]].reset_index(drop=True)
print(f"Dropped total: {df['drop_image'].sum()} (dup: {(df['exact_duplicate']|df['near_duplicate']).sum()}, web-flagged: {df['possible_web_source'].notna().sum()})")
print(f"After cleaning: {len(clean)} images remain")
print("Width stats:\n", clean["width"].describe()[["min", "max", "50%"]])
print("Height stats:\n", clean["height"].describe()[["min", "max", "50%"]])

Dropped total: 134 (dup: 124, web-flagged: 10)
After cleaning: 2164 images remain
Width stats:
 min      41.0
max    2657.0
50%     196.0
Name: width, dtype: float64
Height stats:
 min      41.0
max    2513.0
50%     159.0
Name: height, dtype: float64


In [11]:
# Step 10: Resize and normalize all cleaned images, save as single X/y arrays (split will be done later)
X = np.zeros((len(clean), IMG_SIZE[1], IMG_SIZE[0], 3), dtype=np.float32)
for i, (_, row) in enumerate(clean.iterrows()):
    with Image.open(row["path"]) as im:
        im = im.convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
        X[i] = np.asarray(im, dtype=np.float32) / 255.0
y = clean["label"].to_numpy()

np.save(OUTPUT_DIR / "X_all.npy", X)
np.save(OUTPUT_DIR / "y_all.npy", y)
print(f"Saved: X{X.shape}, y{y.shape}")

Saved: X(2164, 224, 224, 3), y(2164,)


In [12]:
# Step 11: Save full metadata (including dropped/flagged images) and label map for reproducibility
df.to_csv(OUTPUT_DIR / "metadata.csv", index=False)
pd.DataFrame(list(LABEL_MAP.items()), columns=["class_name", "label"]).to_csv(OUTPUT_DIR / "label_map.csv", index=False)
print("Cleaned dataset class counts:")
print(clean["class_name"].value_counts())
print("Done!")

Cleaned dataset class counts:
class_name
Normal            647
Cataract          491
Eyelid            466
Conjunctivitis    347
Uveitis           213
Name: count, dtype: int64
Done!


In [13]:
# Step 12: Check class imbalance on the full cleaned dataset and compute class weights
from sklearn.utils.class_weight import compute_class_weight

class_counts = clean["class_name"].value_counts()
print("Class counts:\n", class_counts)
max_count = class_counts.max()
min_count = class_counts.min()
imbalance_ratio = max_count / min_count
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}x")
print(f"Largest class: {class_counts.idxmax()} ({max_count})")
print(f"Smallest class: {class_counts.idxmin()} ({min_count})")
if imbalance_ratio > 2.0:
    print("\n[WARN] Significant class imbalance detected (>2x) — consider class_weight or oversampling.")
elif imbalance_ratio > 1.3:
    print("\n[INFO] Mild imbalance — class_weight recommended but not critical.")
else:
    print("\nClasses are reasonably balanced.")

y_all = clean["label"].to_numpy()
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_all), y=y_all)
class_weight_dict = {int(k): float(v) for k, v in zip(np.unique(y_all), class_weights)}
print("\nSuggested class_weight dict (for model.fit):")
print(class_weight_dict)

Class counts:
 class_name
Normal            647
Cataract          491
Eyelid            466
Conjunctivitis    347
Uveitis           213
Name: count, dtype: int64

Imbalance ratio (max/min): 3.04x
Largest class: Normal (647)
Smallest class: Uveitis (213)

[WARN] Significant class imbalance detected (>2x) — consider class_weight or oversampling.

Suggested class_weight dict (for model.fit):
{0: 0.8814663951120163, 1: 1.2472622478386166, 2: 0.9287553648068669, 3: 0.6689335394126739, 4: 2.031924882629108}
